# Week 5: Andela Engineering Program – Setup Assistant(Agentic RAG)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/drive/1K2YsX7vPwfqqpQ-89qJ4Eh4EGZzZXsdU?usp=sharing)

RAG Q&A over the **Andela knowledge base** (meeting transcripts). The knowledge base is **downloaded from Google Drive** when not present locally—no need to check data into the repo. **Setup** (Solidroad, GitHub, Copilot, PRs, Udemy, OpenRouter) and **teachings from the deep dives**. Evaluation tab: LLM-as-judge metrics and graphs. Runs locally and on Colab.


In [ ]:
!pip install -q langchain langchain-community langchain-openai langchain-text-splitters langchain-core python-dotenv gradio faiss-cpu sentence-transformers gdown

In [ ]:
import os
import re
from pathlib import Path

# Colab: use secrets; local: use .env
try:
    from google.colab import userdata
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if not IN_COLAB:
    from dotenv import load_dotenv
    load_dotenv(override=True)

def get_openrouter_api_key():
    if IN_COLAB:
        return userdata.get("OPENROUTER_API_KEY")
    return os.getenv("OPENROUTER_API_KEY")

OPENROUTER_API_KEY = get_openrouter_api_key()
if not OPENROUTER_API_KEY:
    print("Warning: OPENROUTER_API_KEY not set. Set it in .env (local) or Colab Secrets.")
else:
    print(f"OpenRouter API key loaded (starts with {OPENROUTER_API_KEY[:8]}...)")

In [ ]:
CONTRIBUTION_DIR = "Wakanda-Chimobi-Ogudu-Week-5"
def find_knowledge_base_dir():
    candidates = [
        Path.cwd() / "andela-knowledge-base",
        Path.cwd() / "week5" / "community-contributions" / CONTRIBUTION_DIR / "andela-knowledge-base",
        Path.cwd() / "community-contributions" / CONTRIBUTION_DIR / "andela-knowledge-base",
        Path.cwd() / "week5" / "andela-knowledge-base",
    ]
    for p in candidates:
        if p.exists():
            return p
    return Path.cwd() / "andela-knowledge-base"

KNOWLEDGE_BASE_DIR = find_knowledge_base_dir()
print(f"Knowledge base path: {KNOWLEDGE_BASE_DIR}")

MODEL_ANSWER = "google/gemini-2.0-flash-001"
MODEL_JUDGE = "anthropic/claude-3.5-sonnet"
OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"

In [ ]:
import zipfile
import urllib.request

GDRIVE_FILE_ID = "1275UGuKfQV0Mel_76-iLaWjUXbv4PLuw"

def ensure_knowledge_base_downloaded():
    global KNOWLEDGE_BASE_DIR
    md_files = list(KNOWLEDGE_BASE_DIR.glob("**/*.md")) if KNOWLEDGE_BASE_DIR.exists() else []
    if md_files:
        print(f"Knowledge base already present at {KNOWLEDGE_BASE_DIR} ({len(md_files)} .md files).")
        return
    if GDRIVE_FILE_ID.startswith("YOUR_"):
        raise ValueError(
            "Knowledge base not found locally. Set GDRIVE_FILE_ID in this cell to a Google Drive file ID "
            "(zip of andela-knowledge-base folder, shared with 'Anyone with the link'). "
            "Or place andela-knowledge-base/ in the repo for local runs."
        )
    # Download zip from Google Drive
    download_dir = Path.cwd() / "andela-knowledge-base"
    download_dir.mkdir(parents=True, exist_ok=True)
    zip_path = Path.cwd() / "andela-knowledge-base.zip"
    url = f"https://drive.google.com/uc?export=download&id={GDRIVE_FILE_ID}"
    print("Downloading andela-knowledge-base from Google Drive...")
    import gdown
    gdown.download(url, str(zip_path), quiet=False, fuzzy=True)
    if not zip_path.exists() or zip_path.stat().st_size == 0:
        raise RuntimeError("Download failed or file empty. Check GDRIVE_FILE_ID and sharing settings.")
    with zipfile.ZipFile(zip_path, "r") as z:
        z.extractall(Path.cwd())
    zip_path.unlink()
    KNOWLEDGE_BASE_DIR = Path.cwd() / "andela-knowledge-base"
    print(f"Knowledge base extracted to {KNOWLEDGE_BASE_DIR}")

ensure_knowledge_base_downloaded()

In [ ]:
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_openai import ChatOpenAI

def load_md_documents(data_dir):
    loader = DirectoryLoader(
        str(data_dir),
        glob="**/*.md",
        loader_cls=TextLoader,
        loader_kwargs={"encoding": "utf-8"},
    )
    return loader.load()

documents = load_md_documents(KNOWLEDGE_BASE_DIR)
print(f"Loaded {len(documents)} document(s) from {KNOWLEDGE_BASE_DIR}")

In [ ]:
# Split into chunks for retrieval
CHUNK_SIZE = 800
CHUNK_OVERLAP = 120
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    length_function=len,
    separators=["\n\n", "\n", " ", ""],
)
split_docs = text_splitter.split_documents(documents)
print(f"Split into {len(split_docs)} chunks")

In [ ]:
# Embeddings (local, no API key) and vector store
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={"device": "cpu"},
)
vector_store = FAISS.from_documents(split_docs, embedding_model)
retriever = vector_store.as_retriever(search_kwargs={"k": 5})

In [ ]:
# LLM for answering (OpenRouter, lower frontier)
llm_answer = ChatOpenAI(
    model=MODEL_ANSWER,
    openai_api_key=OPENROUTER_API_KEY,
    openai_api_base=OPENROUTER_BASE_URL,
    temperature=0.2,
)

# Prompt: setup + deep-dive teachings; answer only from context
RAG_SYSTEM_PROMPT = """You are an assistant for students in the Andela Engineering program. You help with (1) setup and onboarding: Solidroad, GitHub, Copilot, pull requests (PRs), Udemy, OpenRouter, and related tools or processes; and (2) questions about what was taught in the technical and expert deep dive sessions (e.g. vector embeddings, RAG, recommendations, or other topics covered in those meetings).
Answer ONLY using the following context from meeting transcripts and program materials.
If the context does not contain enough information to answer the question, respond with exactly: I don't have the answer.
Do not make up information. Keep answers concise and practical.

Context:
{context}

Question: {input}

Answer:"""

rag_prompt = ChatPromptTemplate.from_messages([
    ("system", RAG_SYSTEM_PROMPT),
])

In [ ]:
def format_docs(docs):
    if not docs:
        return "No relevant context found."
    return "\n\n---\n\n".join(doc.page_content for doc in docs)

# RAG chain: retrieve -> format context -> prompt -> LLM -> string
rag_chain = (
    {"context": retriever | format_docs, "input": RunnablePassthrough()}
    | rag_prompt
    | llm_answer
    | StrOutputParser()
)

In [ ]:
def answer_question(question: str) -> str:    
    if not question or not question.strip():
        return "Please ask a question about setup (e.g. Solidroad, GitHub, Copilot, PRs, Udemy, OpenRouter) or about what was taught in the deep dives."
    try:
        return rag_chain.invoke(question.strip())
    except Exception as e:
        return f"Error: {str(e)}"

# Quick test (setup-related)
test_question = "How do I set up GitHub or OpenRouter for the program?"
test_answer = answer_question(test_question)
print("Test question:", test_question)
print("Test answer:", test_answer[:400] + "..." if len(test_answer) > 400 else test_answer)

In [ ]:
import gradio as gr

In [ ]:
def chat_respond(message, chat_history):
    if not message or not message.strip():
        return chat_history, ""
    reply = answer_question(message)
    new_history = chat_history + [{"role": "user", "content": message}, {"role": "assistant", "content": reply}]
    return new_history, ""

In [ ]:
# Judge LLM (higher frontier) for evaluation
llm_judge = ChatOpenAI(
    model=MODEL_JUDGE,
    openai_api_key=OPENROUTER_API_KEY,
    openai_api_base=OPENROUTER_BASE_URL,
    temperature=0.0,
)

JUDGE_PROMPT = """You are an impartial judge evaluating an AI assistant's answer for a student question.
Question: {question}
Context (retrieved knowledge base excerpts): {context}
Assistant's answer: {answer}

Score from 1 to 5 (integer only):
- Faithfulness (1-5): Is the answer fully supported by the context? 1=not at all, 5=fully grounded.
- Relevance (1-5): Does the answer address the question? 1=off-topic, 5=fully addresses it.

Respond with exactly two lines:
FAITHFULNESS: <number>
RELEVANCE: <number>"""

judge_prompt_template = ChatPromptTemplate.from_messages([("human", JUDGE_PROMPT)])
judge_chain = judge_prompt_template | llm_judge | StrOutputParser()

In [ ]:
def parse_judge_scores(text):    
    faithfulness_score = 3
    relevance_score = 3
    for line in text.strip().upper().split("\n"):
        line = line.strip()
        if line.startswith("FAITHFULNESS:"):
            try:
                faithfulness_score = int(re.search(r"\d+", line).group())
                faithfulness_score = max(1, min(5, faithfulness_score))
            except (ValueError, AttributeError):
                pass
        elif line.startswith("RELEVANCE:"):
            try:
                relevance_score = int(re.search(r"\d+", line).group())
                relevance_score = max(1, min(5, relevance_score))
            except (ValueError, AttributeError):
                pass
    return faithfulness_score, relevance_score

In [ ]:
# Default evaluation: setup (Solidroad, GitHub, Copilot, PRs, Udemy, OpenRouter) + deep-dive teachings
DEFAULT_EVAL_QUESTIONS = [
    "How do I get set up on Solidroad for the Andela program?",
    "What do I need to do with GitHub for this bootcamp?",
    "How do I use or set up Copilot for the course?",
    "How do I submit or create a pull request (PR) for my work?",
    "Where do I find or access the Udemy course content?",
    "How do I get or use an OpenRouter API key for the program?",
    "What setup steps were mentioned in the kick-off or office hours?",
    "What was taught about vector embeddings in the technical deep dive?",
]

In [ ]:
def run_evaluation(questions_list_text):    
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt

    lines = [q.strip() for q in questions_list_text.strip().split("\n") if q.strip()]
    questions = lines if lines else DEFAULT_EVAL_QUESTIONS

    results = []
    all_faithfulness = []
    all_relevance = []

    for question in questions:
        context = format_docs(retriever.invoke(question))
        model_answer = answer_question(question)
        judge_input = {"question": question, "context": context[:3000], "answer": model_answer}
        try:
            judge_output = judge_chain.invoke(judge_input)
            faith, rel = parse_judge_scores(judge_output)
        except Exception:
            faith, rel = 3, 3
        all_faithfulness.append(faith)
        all_relevance.append(rel)
        results.append((question[:60] + "..." if len(question) > 60 else question, model_answer[:150] + "..." if len(model_answer) > 150 else model_answer, faith, rel))

    avg_faithfulness = sum(all_faithfulness) / len(all_faithfulness) if all_faithfulness else 0
    avg_relevance = sum(all_relevance) / len(all_relevance) if all_relevance else 0
    avg_overall = (avg_faithfulness + avg_relevance) / 2

    # Bar chart
    fig, ax = plt.subplots(figsize=(6, 4))
    metrics_names = ["Faithfulness", "Relevance", "Overall"]
    metrics_values = [avg_faithfulness, avg_relevance, avg_overall]
    bars = ax.bar(metrics_names, metrics_values, color=["#2ecc71", "#3498db", "#9b59b6"])
    ax.set_ylim(0, 5.5)
    ax.set_ylabel("Score (1-5)")
    ax.set_title("Evaluation metrics (LLM-as-judge)")
    for bar, val in zip(bars, metrics_values):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.1, f"{val:.2f}", ha="center", fontsize=10)
    plt.tight_layout()

    # Table as markdown
    table_header = "| Question | Answer (excerpt) | Faithfulness | Relevance |\n|----------|------------------|--------------|----------|\n"
    table_rows = "".join(f"| {r[0]} | {r[1]} | {r[2]} | {r[3]} |\n" for r in results)
    results_md = "## Per-question scores\n\n" + table_header + table_rows + f"\n**Averages:** Faithfulness={avg_faithfulness:.2f}, Relevance={avg_relevance:.2f}, Overall={avg_overall:.2f}"

    return results_md, fig

In [ ]:
# Build Gradio app: Q&A + Evaluation
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning, module="gradio")
with gr.Blocks(title="Andela Engineering – Setup Assistant", theme=gr.themes.Soft()) as demo:
    gr.Markdown("# Andela Engineering Program – Setup Assistant")
    gr.Markdown("Ask **setup** questions (Solidroad, GitHub, Copilot, PRs, Udemy, OpenRouter) or questions about **teachings in the deep dives**. Answers are grounded in meeting transcripts. Evaluation tab: run test questions and LLM-as-judge metrics.")

    with gr.Tabs():
        with gr.TabItem("Q&A"):
            chatbot = gr.Chatbot(label="Chat", height=400, type="messages", allow_tags=False)
            msg_input = gr.Textbox(label="Your question", placeholder="e.g. How do I set up OpenRouter? Or: What was taught about vector embeddings in the deep dive?")
            with gr.Row():
                submit_btn = gr.Button("Submit")
                clear_btn = gr.ClearButton([msg_input, chatbot])

            msg_input.submit(fn=chat_respond, inputs=[msg_input, chatbot], outputs=[chatbot, msg_input])
            submit_btn.click(fn=chat_respond, inputs=[msg_input, chatbot], outputs=[chatbot, msg_input])

        with gr.TabItem("Evaluation"):
            eval_questions = gr.Textbox(
                label="Setup evaluation questions (one per line; default covers Solidroad, GitHub, Copilot, PRs, Udemy, OpenRouter)",
                value="\n".join(DEFAULT_EVAL_QUESTIONS),
                lines=10,
            )
            run_eval_btn = gr.Button("Run evaluation")
            eval_results_md = gr.Markdown(label="Results")
            eval_plot = gr.Plot(label="Metrics")

            def run_eval_and_show(questions_text):
                md, fig = run_evaluation(questions_text)
                return md, fig

            run_eval_btn.click(
                fn=run_eval_and_show,
                inputs=[eval_questions],
                outputs=[eval_results_md, eval_plot],
            )

    # In Colab: server_name for reachability
    demo.launch(debug=False, share=True, server_name="0.0.0.0" if IN_COLAB else None)